# Normalize single-cell profiles

> NOTE: We normalize single-cells to the whole plate, given that the sample size is much larger and less likely as impacted by variation.

## Import libraries

In [1]:
import csv
import gc
import os
import pathlib
import pprint
import time
from datetime import datetime, timezone

import pandas as pd
from pycytominer import normalize

## Set paths and variables

In [2]:
# Directory with the converted profiles (one folder per cell line, one parquet file per plate/time point)
converted_dir = pathlib.Path("./data/converted_profiles").resolve(strict=True)

# extract the plate names from the converted profiles (<plate>_<cell_line>_converted.parquet)
plate_names = sorted(
    {file.name.split("_")[0] for file in converted_dir.glob("*/*_converted.parquet")}
)

# Directory containing per-plate annotated profiles from 2.annotate.ipynb
annotated_dir = pathlib.Path("./data/annotated_profiles")

# output path for normalized single-cell profiles
output_dir = pathlib.Path("./data/single_cell_profiles")
output_dir.mkdir(parents=True, exist_ok=True)

plate_names

['BR00150695',
 'BR00150696',
 'BR00150697',
 'BR00150698',
 'BR00150699',
 'BR00150700',
 'BR00150701',
 'BR00150702',
 'BR00150703']

## Set dictionary with plates to process

In [3]:
# Create plate info dictionary
plate_info_dictionary = {
    plate_id: {
        # Annotated profiles are produced per-plate by 2.annotate.ipynb
        "annotated_path": (
            str((annotated_dir / f"{plate_id}_annotated.parquet").resolve())
            if (annotated_dir / f"{plate_id}_annotated.parquet").exists()
            else None
        ),
    }
    for plate_id in plate_names
}

# Display the dictionary to verify the entries
pprint.pprint(plate_info_dictionary, indent=4)

{   'BR00150695': {   'annotated_path': '/media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/4.preprocess_features/data/annotated_profiles/BR00150695_annotated.parquet'},
    'BR00150696': {   'annotated_path': '/media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/4.preprocess_features/data/annotated_profiles/BR00150696_annotated.parquet'},
    'BR00150697': {   'annotated_path': '/media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/4.preprocess_features/data/annotated_profiles/BR00150697_annotated.parquet'},
    'BR00150698': {   'annotated_path': '/media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/4.preprocess_features/data/annotated_profiles/BR00150698_annotated.parquet'},
    'BR00150699': {   'annotated_path': '/media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/4.preprocess_features/data/annotated_profiles/BR00150699_annotated.parquet'},
    'BR00150700': {   'annotated_path': '/media/18tbdrive/1.Github_Rep

In [4]:
# Run configuration, both driven by environment variables so run_preprocessing_pipeline.sh
# can control them without papermill:
#   PLATE_ID  - restrict to a single plate (used to run each plate as its own
#               process so memory doesn't accumulate across plates). Leave
#               unset to process all plates, e.g. when running interactively.
#   OVERWRITE - if set (1/true/yes), reprocess and overwrite a plate's output
#               even if it already exists. Leave unset (the default) to skip
#               plates that already have output.
plate_id_filter = os.environ.get("PLATE_ID")
if plate_id_filter:
    if plate_id_filter not in plate_info_dictionary:
        raise ValueError(f"Unknown plate_id in PLATE_ID env var: {plate_id_filter}")
    plate_info_dictionary = {plate_id_filter: plate_info_dictionary[plate_id_filter]}

overwrite = os.environ.get("OVERWRITE", "").strip().lower() in ("1", "true", "yes")

plate_info_dictionary

{'BR00150695': {'annotated_path': '/media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/4.preprocess_features/data/annotated_profiles/BR00150695_annotated.parquet'},
 'BR00150696': {'annotated_path': '/media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/4.preprocess_features/data/annotated_profiles/BR00150696_annotated.parquet'},
 'BR00150697': {'annotated_path': '/media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/4.preprocess_features/data/annotated_profiles/BR00150697_annotated.parquet'},
 'BR00150698': {'annotated_path': '/media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/4.preprocess_features/data/annotated_profiles/BR00150698_annotated.parquet'},
 'BR00150699': {'annotated_path': '/media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/4.preprocess_features/data/annotated_profiles/BR00150699_annotated.parquet'},
 'BR00150700': {'annotated_path': '/media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profilin

## Normalize profiles

In [5]:
timing_log_path = output_dir / "timing_log_normalize.csv"

for plate_id, info in plate_info_dictionary.items():
    if info["annotated_path"] is None:
        print(
            f"Skipping {plate_id}: not yet annotated "
            "(run 2.annotate.ipynb for this plate first)"
        )
        continue

    output_normalized_file = str(output_dir / f"{plate_id}_sc_normalized.parquet")

    # Already normalized, so this plate can safely be skipped on a rerun
    # (unless OVERWRITE is set, in which case it's reprocessed regardless)
    if pathlib.Path(output_normalized_file).exists() and not overwrite:
        print(f"Skipping {plate_id}: already normalized (found {output_normalized_file})")
        continue

    print(f"Normalizing {plate_id}")
    plate_start_time = time.time()

    # Load the already-annotated single-cell profile (shared with
    # 3.bulk_processing.ipynb, so annotate() only ran once for this plate)
    annotated_df = pd.read_parquet(info["annotated_path"])

    normalize_start_time = time.time()
    normalize(
        profiles=annotated_df,
        method="standardize",  # use standardize method as default
        output_file=output_normalized_file,
        output_type="parquet",
        samples="all",  # apply normalization based on all samples
    )
    normalize_seconds = time.time() - normalize_start_time

    # Clear memory
    del annotated_df
    gc.collect()

    total_seconds = time.time() - plate_start_time
    print(
        f"Normalization completed for {plate_id}! "
        f"(normalize={normalize_seconds:.1f}s, total={total_seconds:.1f}s)"
    )

    # Record timing so per-plate performance can be compared across runs
    write_header = not timing_log_path.exists()
    with open(timing_log_path, "a", newline="") as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(["plate_id", "normalize_seconds", "total_seconds", "timestamp"])
        writer.writerow(
            [
                plate_id,
                f"{normalize_seconds:.1f}",
                f"{total_seconds:.1f}",
                datetime.now(timezone.utc).isoformat(timespec="seconds"),
            ]
        )

Normalizing BR00150695


Normalization completed for BR00150695! (normalize=13.8s, total=15.7s)
Normalizing BR00150696


Normalization completed for BR00150696! (normalize=16.5s, total=18.6s)
Normalizing BR00150697


Normalization completed for BR00150697! (normalize=62.2s, total=64.6s)
Normalizing BR00150698


Normalization completed for BR00150698! (normalize=26.2s, total=28.4s)
Normalizing BR00150699


Normalization completed for BR00150699! (normalize=41.1s, total=43.8s)
Normalizing BR00150700


Normalization completed for BR00150700! (normalize=51.6s, total=55.0s)
Normalizing BR00150701


Normalization completed for BR00150701! (normalize=19.5s, total=21.1s)
Normalizing BR00150702


Normalization completed for BR00150702! (normalize=25.6s, total=27.8s)
Normalizing BR00150703


Normalization completed for BR00150703! (normalize=66.1s, total=69.5s)
